# ЛР1. Ручний backpropagation: дослідження

У цьому блокноті покроково перевіряємо ручну реалізацію мережі на Iris. Спочатку перевіряємо дані та проміжні значення, потім порівнюємо градієнти з PyTorch і рахуємо числові похідні. Також проводимо дослід із навмисно пропущеним діленням на розмір вибірки.

Основні функції реалізовані у файлі lab1.py.

## 0. Імпорти та відтворюваність

Підключаємо необхідні бібліотеки та модуль лабораторної роботи.

In [ ]:
from pathlib import Path
import sys
import numpy as np

# додаємо шлях до папки з кодом
here = Path.cwd()
lab_dir = here if (here / 'lab1.py').exists() else here / 'Lab1'
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

from lab1 import (
    backward,
    compare_gradients,
    finite_difference_checks,
    forward,
    forward_logits,
    initialize,
    prepare_data,
    torch_reference,
)

np.set_printoptions(precision=6, suppress=True)
print('Використовую код із:', lab_dir / 'lab1.py')

## 1. Дані та фіксований поділ

Для кожного з трьох класів беремо 35 об'єктів у train і 15 у test після перемішування генератором `np.random.default_rng(0)`. Статистики стандартизації обчислюються тільки на train. Очікую форми `(105, 4)` і `(45, 4)` та однакові кількості класів.

In [2]:
X_train, y_train, X_test, y_test = prepare_data()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')
print('train class counts:', np.bincount(y_train).tolist())
print('test class counts: ', np.bincount(y_test).tolist())
print('dtype:', X_train.dtype)

X_train: (105, 4), y_train: (105,)
X_test:  (45, 4), y_test:  (45,)
train class counts: [35, 35, 35]
test class counts:  [15, 15, 15]
dtype: float64


## 2. Початкові параметри та forward pass

Ваги ініціалізовані послідовно одним генератором: `W1` за He, `W2` за Xavier, зміщення нульові. У кеші зберігаються `Z1`, `A1` і `P`. Суми рядків `P` мають бути одиницею, а loss — скінченним.

In [3]:
parameters = initialize()
loss_numpy, cache = forward(X_train, y_train, parameters)
z1, a1, probabilities = cache
logits = forward_logits(X_train, parameters)

print('loss:', loss_numpy)
print(f'Z1 shape: {z1.shape}, A1 shape: {a1.shape}, P shape: {probabilities.shape}')
row_sums = probabilities.sum(axis=1)
print('probability row sums:', row_sums.min(), '...', row_sums.max())
print('logit range:', logits.min(), '...', logits.max())
test_predictions = np.argmax(forward_logits(X_test, parameters), axis=1)
print('initial test accuracy:', np.mean(test_predictions == y_test))

loss: 1.4562007801142816
Z1 shape: (105, 8), A1 shape: (105, 8), P shape: (105, 3)
probability row sums: 0.9999999999999998 ... 1.0000000000000002
logit range: -1.735111262228266 ... 3.8428135249706736
initial test accuracy: 0.222222222222


## 3. Що саме диференціюємо

Для `N=105` ручний backward використовує:

```text
D2 = (P - one_hot(y)) / N
dW2 = A1.T @ D2,       db2 = D2.sum(axis=0)
D1 = (D2 @ W2.T) * (Z1 > 0)
dW1 = X.T @ D1,        db1 = D1.sum(axis=0)
```

Тому потрібні саме ті проміжні значення, які повертає `forward()`: `Z1` для похідної ReLU, `A1` для `dW2` і `P` для `D2`; `X` передається в `backward()` окремо.

In [4]:
manual_gradients = backward(X_train, y_train, parameters, cache)
for name in ('W1', 'b1', 'W2', 'b2'):
    value = getattr(manual_gradients, name)
    print(f'{name} gradient shape={value.shape}, norm={np.linalg.norm(value):.6f}')

W1 gradient shape=(4, 8), norm=0.418334
b1 gradient shape=(8,), norm=0.229438
W2 gradient shape=(8, 3), norm=0.734514
b2 gradient shape=(3,), norm=0.139269


## 4. Звірка з PyTorch autograd

PyTorch використовує ті самі дані та параметри. При копіюванні транспоную ваги, бо `torch.nn.Linear` зберігає їх як `(out_features, in_features)`. Для проходження потрібно, щоб кожна максимальна різниця була не більшою за `10⁻¹²`.

In [5]:
torch_loss, torch_gradients = torch_reference(X_train, y_train, parameters)
gradient_comparison = compare_gradients(manual_gradients, torch_gradients)
loss_difference = abs(loss_numpy - torch_loss)
print(f'loss NumPy:   {loss_numpy:.16g}')
print(f'loss PyTorch: {torch_loss:.16g}')
print(f'loss max abs difference: {loss_difference:.12g} -> {loss_difference <= 1e-12}')
for name, (difference, passed) in gradient_comparison.items():
    print(f'{name} max abs difference: {difference:.12g} -> {passed}')

loss NumPy:   1.456200780114282
loss PyTorch: 1.456200780114282
loss max abs difference: 2.22044604925e-16 -> True
W1 max abs difference: 2.77555756156e-17 -> True
b1 max abs difference: 3.29597460436e-17 -> True
W2 max abs difference: 1.11022302463e-16 -> True
b2 max abs difference: 1.2490009027e-16 -> True


## 5. Чисельна перевірка окремих елементів

Змінюємо один параметр на плюс і мінус 1e-6 та обчислюємо центральну різницю. Критерій проходження: абсолютна різниця не більше 1e-7.

In [6]:
finite_rows = finite_difference_checks(
    X_train, y_train, parameters, manual_gradients
)
print('parameter | backward | numerical | abs difference | passed')
for name, analytic, numerical, difference, passed in finite_rows:
    print(
        f'{name} | {analytic:.12g} | {numerical:.12g} | '
        f'{difference:.12g} | {passed}'
    )

parameter | backward | numerical | abs difference | passed
W1(0, 0) | -0.0199201843306 | -0.0199201843909 | 6.03427273427e-11 | True
b1(0,) | -0.0705661115844 | -0.0705661116207 | 3.63268304326e-11 | True
W2(0, 0) | 0.135230208605 | 0.135230208698 | 9.21804022003e-11 | True
b2(0,) | 0.085521644842 | 0.085521644988 | 1.46028980574e-10 | True


## 6. Дослід із навмисною помилкою

Гіпотеза перед запуском:

- втрата не зміниться, оскільки прямий прохід залишається тим самим;
- кожен ненульовий градієнт збільшиться у 105 разів через відсутність ділення на N;
- перевірка з PyTorch та числові похідні вкажуть на помилку, а перевірка втрати пройде.

Параметри між експериментами не оновлюємо.

In [7]:
bug_gradients = backward(
    X_train, y_train, parameters, cache, bug=True
)
bug_loss, _ = forward(X_train, y_train, parameters)
bug_comparison = compare_gradients(bug_gradients, torch_gradients)
bug_rows = finite_difference_checks(
    X_train, y_train, parameters, bug_gradients
)

print('loss remains unchanged:', bug_loss == loss_numpy)
bug_loss_difference = abs(bug_loss - torch_loss)
print(f'bug loss vs PyTorch: {bug_loss_difference:.12g} -> {bug_loss_difference <= 1e-12}')
for name, (difference, passed) in bug_comparison.items():
    good = getattr(manual_gradients, name)
    bad = getattr(bug_gradients, name)
    active = np.abs(good) > 1e-14
    ratio = np.mean(bad[active] / good[active])
    print(f'{name}: mean ratio {ratio:.1f}, PyTorch check -> {passed}')
for name, _, _, difference, passed in bug_rows:
    print(f'{name} numerical check -> {passed} (abs difference {difference:.12g})')

loss remains unchanged: True
bug loss vs PyTorch: 2.22044604925e-16 -> True
W1: mean ratio 105.0, PyTorch check -> False
b1: mean ratio 105.0, PyTorch check -> False
W2: mean ratio 105.0, PyTorch check -> False
b2: mean ratio 105.0, PyTorch check -> False
W1(0, 0) numerical check -> False (abs difference 2.07169917032)
b1(0,) numerical check -> False (abs difference 7.33887560474)
W2(0, 0) numerical check -> False (abs difference 14.0639416949)
b2(0,) numerical check -> False (abs difference 8.89425106342)


## Висновок

Ручний розрахунок градієнтів збігається з PyTorch до машинної точності, а числові похідні підтверджують правильність формул. Дослід із помилкою підтвердив очікування: втрата не змінилася, але всі градієнти зросли у 105 разів.